In [5]:
!pip install -q scapy

from google.colab import drive
import scapy.all as scapy
import pandas as pd
import pickle
import os
import glob

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/man-bot-detection/data/raw'
print(f'Base directory: {BASE_DIR}')

def find_pcap(folder):
    pcaps = glob.glob(os.path.join(folder, '*.pcap')) + glob.glob(os.path.join(folder, '*.pcapng'))
    return pcaps[0] if pcaps else None

def load_labels_ctu(folder):
    labels = {}
    attack = os.path.join(folder, 'CTU13_Attack_Traffic.csv')
    normal = os.path.join(folder, 'CTU13_Normal_Traffic.csv')
    if os.path.exists(attack):
        df = pd.read_csv(attack)
        for _, row in df.iterrows():
            key = (row['SrcAddr'], row['Sport'], row['DstAddr'], row['Dport'])
            labels[key] = 'attack'
            labels[(row['DstAddr'], row['Dport'], row['SrcAddr'], row['Sport'])] = 'attack'
        print(f'Loaded {len(df)} attack labels from {attack}')
    if os.path.exists(normal):
        df = pd.read_csv(normal)
        for _, row in df.iterrows():
            key = (row['SrcAddr'], row['Sport'], row['DstAddr'], row['Dport'])
            labels[key] = 'normal'
            labels[(row['DstAddr'], row['Dport'], row['SrcAddr'], row['Sport'])] = 'normal'
        print(f'Loaded {len(df)} normal labels from {normal}')
    return labels

def load_labels_iot(folder):
    labels = {}
    log = os.path.join(folder, 'mirai_big.log')
    if os.path.exists(log):
        df = pd.read_csv(log, sep='\t')
        for _, row in df.iterrows():
            key = (row['id.orig_h'], row['id.orig_p'], row['id.resp_h'], row['id.resp_p'])
            labels[key] = row['label']
            labels[(row['id.resp_h'], row['id.resp_p'], row['id.orig_h'], row['id.orig_p'])] = row['label']
        print(f'Loaded {len(df)} labels from {log}')
    return labels

def read_pcap(pcap_path):
    flows = {}
    if not os.path.exists(pcap_path):
        print(f'PCAP file not found: {pcap_path}')
        return flows
    print(f'Reading PCAP: {pcap_path}')
    count = 0
    for pkt in scapy.PcapReader(pcap_path):
        if not pkt.haslayer(scapy.IP) or not pkt.haslayer(scapy.TCP):
            continue
        ip = pkt[scapy.IP]
        tcp = pkt[scapy.TCP]
        key = (ip.src, tcp.sport, ip.dst, tcp.dport)
        flows.setdefault(key, []).append({
            'time': pkt.time,
            'size': len(pkt),
            'direction': 1 if ip.src < ip.dst else -1,
            'flags': str(tcp.flags)
        })
        count += 1
        if count % 10000 == 0:
            print(f'Processed {count} packets, {len(flows)} flows so far')
    print(f'Finished reading {count} packets, found {len(flows)} flows')
    return flows

def extract_sequences(flows, labels, window_sec=120, max_len=500):
    result = {}
    for key, packets in flows.items():
        if not packets:
            continue
        label = labels.get(key, 'unknown')
        if label == 'unknown':
            continue
        first_time = packets[0]['time']
        window_end = first_time + window_sec
        seq_packets = [p for p in packets if p['time'] <= window_end][:max_len]
        seq = []
        prev = None
        for p in seq_packets:
            iat = p['time'] - prev if prev is not None else 0.0
            seq.append({'iat': iat, 'size': p['size'], 'direction': p['direction'], 'flags': p['flags']})
            prev = p['time']
        if seq:
            result[key] = {'sequence': seq, 'label': label}
    return result

def process_dataset(folder, label_type):
    print(f'\nProcessing {folder}...')
    pcap = find_pcap(folder)
    if pcap is None:
        print(f'No pcap file found in {folder}')
        return {}
    print(f'Found PCAP: {pcap}')
    flows = read_pcap(pcap)
    if not flows:
        print('No flows extracted')
        return {}
    print(f'Loading labels for {label_type}...')
    labels = load_labels_ctu(folder) if label_type == 'ctu' else load_labels_iot(folder)
    print(f'Total labels loaded: {len(labels)}')
    result = extract_sequences(flows, labels)
    print(f'Extracted {len(result)} sequences with labels')
    return result

all_data = {}
for folder_name, label_type in [('ctu13', 'ctu'), ('iot23', 'iot')]:
    folder_path = os.path.join(BASE_DIR, folder_name)
    if os.path.exists(folder_path):
        print(f'Folder exists: {folder_path}')
        data = process_dataset(folder_path, label_type)
        all_data.update(data)
        print(f'Added {len(data)} sequences from {folder_name}')
    else:
        print(f'Folder does not exist: {folder_path}')

print(f'\nTotal sequences collected: {len(all_data)}')

if all_data:
    output_path = '/content/drive/MyDrive/man-bot-detection/sequences.pkl'
    with open(output_path, 'wb') as f:
        pickle.dump(all_data, f)
    print(f'Successfully saved to {output_path}')
else:
    print('No data to save. Check your folder structure and files.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Base directory: /content/drive/MyDrive/man-bot-detection/data/raw
Folder exists: /content/drive/MyDrive/man-bot-detection/data/raw/ctu13

Processing /content/drive/MyDrive/man-bot-detection/data/raw/ctu13...
No pcap file found in /content/drive/MyDrive/man-bot-detection/data/raw/ctu13
Added 0 sequences from ctu13
Folder exists: /content/drive/MyDrive/man-bot-detection/data/raw/iot23

Processing /content/drive/MyDrive/man-bot-detection/data/raw/iot23...
No pcap file found in /content/drive/MyDrive/man-bot-detection/data/raw/iot23
Added 0 sequences from iot23

Total sequences collected: 0
No data to save. Check your folder structure and files.
